In [ ]:
from google.colab import drive
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from transformers import RobertaModel #Model used
from transformers import RobertaTokenizer
from transformers import AdamW
from sklearn.metrics import f1_score

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Dataset path
dataset_path = '/content/drive/My Drive/SemEval'

In [ ]:
# Load the CSV file
def load_data(file_path,delimeter = ","):
    df = pd.read_csv(file_path, delimiter = delimeter)
    texts = df["text"].tolist()
    labels = df[["anger", "fear", "joy", "sadness", "surprise"]].values
    return texts, labels

# Load train and test data
train_file = "/content/drive/My Drive/SemEval/track_a/train/eng.csv"
test_file = "/content/drive/My Drive/SemEval/track_a/dev/eng.csv"
human_pred = "/content/drive/My Drive/SemEval/track_a/Human_eval_eng.csv"

train_texts, train_labels = load_data(train_file)
test_texts, test_labels = load_data(test_file)
h_texts, h_labels = load_data(human_pred)

# Initialize tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def find_max_length(texts, tokenizer):
    tokenized_texts = [tokenizer.tokenize(text) for text in texts]
    return max(len(tokens) for tokens in tokenized_texts)

max_length = find_max_length(train_texts + test_texts, tokenizer)  # Find max length from both train & test

print(f"Dynamic max length: {max_length}")

def tokenize_texts(texts, tokenizer, max_length):
    return tokenizer(
        texts,
        max_length=max_length,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

train_encodings = tokenize_texts(train_texts, tokenizer, max_length)
test_encodings = tokenize_texts(test_texts, tokenizer, max_length)
h_encodings = tokenize_texts(h_texts, tokenizer, max_length)


EmotionDataset: Custom PyTorch Dataset for Emotion Classification.

- Stores tokenized inputs (`input_ids`, `attention_mask`) and labels (if provided).
- Converts inputs and labels to PyTorch tensors and moves them to the specified device (`cpu` or `cuda`).
- Implements `__len__()` to return the dataset size.
- Implements `__getitem__()` to retrieve a sample in dictionary format.
- Supports labeled data (for training) and unlabeled data (for inference).

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels=None, device="cpu"):
        """
        Custom PyTorch Dataset for Emotion Classification.

        Args:
            encodings (dict): Tokenized input (should contain "input_ids" and "attention_mask").
            labels (list or tensor, optional): Corresponding emotion labels (multi-label format).
            device (str): Device to store tensors ('cpu' or 'cuda').
        """
        self.encodings = {key: torch.tensor(val, dtype=torch.long).to(device) for key, val in encodings.items()}
        self.labels = torch.tensor(labels, dtype=torch.float).to(device) if labels is not None else None

    def __len__(self):

        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):

        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = self.labels[idx]
        return item


train_dataset = EmotionDataset(train_encodings, train_labels, device="cuda")
test_dataset = EmotionDataset(test_encodings, test_labels, device="cuda")
h_dataset = EmotionDataset(h_encodings,test_labels, device="cuda")



RobertaClass: A text classification model based on RoBERTa.

- Uses a pre-trained "roberta-large" model to extract contextual embeddings.
- The [CLS] token representation (pooler_output) is used for classification.
- Three fully connected layers refine the representation:
  - fc1 (1024 → 512) with LayerNorm, GELU activation, and Dropout.
  - fc2 (512 → num_label)


- LayerNorm (Layer Normalization): Normalizes activations across features, stabilizing training and improving convergence.
- Relu is used
- Dropout: Randomly drops neurons during training to prevent overfitting and improve generalization.


In [ ]:
import torch
import torch.nn as nn
import random
import numpy as np
from transformers import RobertaModel

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

class RobertaClass(nn.Module):
    def __init__(self, num_labels=5):
        super(RobertaClass, self).__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-large")
        self.dropout = nn.Dropout(0.1)  # Dropout rate of 0.1

        # Fully connected layers
        self.fc1 = nn.Linear(1024, 512)
        self.layer_norm1 = nn.LayerNorm(512)
        self.fc2 = nn.Linear(512, num_labels)

        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask):

        output = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = output.pooler_output

        x = self.fc1(cls_output)
        x = self.layer_norm1(x)
        x = self.relu(x)
        x = self.dropout(x)


        logits = self.fc2(x)

        return logits


In [ ]:
# Initialize model
num_labels = 5  # 5 emotions
model = RobertaClass(num_labels=num_labels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


DataLoaders for batching and shuffling:

- `BATCH_SIZE = 16`: Defines how many samples per batch.
- `train_loader`: Loads training data with shuffling (`shuffle=True`) to improve generalization.
- `test_loader`: Loads test data without shuffling (`shuffle=False`) to ensure consistent evaluation.
- `DataLoader` handles batching, shuffling, and efficient data loading for training and inference.

In [ ]:

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


Loss Function & Optimizer:

- `BCEWithLogitsLoss()`: Used for multi-label classification. Combines sigmoid activation with binary cross-entropy loss.
- `AdamW`: Optimizer designed for transformers.
  - `lr=1e-5`: Learning rate (controls step size during optimization).
  - `weight_decay=1e-2`: Regularization to prevent overfitting.

In [ ]:
# Loss function for multi-label classification
criterion = nn.BCEWithLogitsLoss()

optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)


Training Loop & Learning Rate Scheduler:

- `device`: Moves model to GPU (if available) for faster training.
- `get_scheduler("linear")`: Implements a linear learning rate decay.
- `epochs = 20`: Trains for 20 iterations over the dataset.
- `num_training_steps`: Total steps based on `epochs * dataset size`.
- `tqdm`: Displays progress bar for each training batch.

Training Process:
1. Set model to `train()` mode.
2. Iterate through `train_loader`, moving data to the device.
3. Compute predictions and calculate loss (`BCEWithLogitsLoss`).
4. Perform backpropagation (`loss.backward()`).
5. Update model weights (`optimizer.step()`).
6. Adjust learning rate (`lr_scheduler.step()`).
7. Print progress and total loss per epoch.

In [ ]:
import torch
from transformers import get_scheduler
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs =20
num_training_steps = epochs * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

loss_history = []

# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1} finished. Average Loss: {avg_loss:.4f}")

# Plot the training loss over epochs
plt.figure(figsize=(8, 6))
plt.plot(range(1, epochs + 1), loss_history, marker='o', linestyle='-', color='b')
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.title('Training Loss per Epoch')
plt.grid(True)
# Save the plot
plot_path = "..path to save/training_loss_plot.png"
plt.savefig(plot_path)

plot_path
plt.show()



Evaluation Function:

- `model.eval()`: Sets the model to evaluation mode (disables dropout & gradient updates).
- `torch.no_grad()`: Disables gradient computation for faster inference.
- Iterates over `test_loader`, moving data to the correct device.
- Computes model outputs and applies `sigmoid()` to get probabilities.
- Converts probabilities to binary predictions using a threshold of 0.5.
- Uses `classification_report()` to generate precision, recall, and F1-score.



In [ ]:
from sklearn.metrics import classification_report
import numpy as np

def evaluate(model, test_loader):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.sigmoid(outputs).cpu().numpy()  # Convert logits to probabilities
            labels = labels.cpu().numpy()

            predictions.extend(preds)
            true_labels.extend(labels)


    predictions = np.array(predictions) > 0.5

    print(classification_report(true_labels, predictions, target_names=["anger", "fear", "joy", "sadness", "surprise"]))

# Evaluate the model
evaluate(model, test_loader)


In [ ]:
model_save_path = "roberta_emotion_model_Relu.pt"
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

In [ ]:
# Evaluation with human predictions
model.eval()
all_preds = []
all_labels = []
test_loss = 0.0
correct = 0
total = 0

h_loader = DataLoader(h_dataset, batch_size=8, shuffle=False)

with torch.no_grad():
    for batch in h_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask)

        preds = (torch.sigmoid(outputs) > 0.5).float()  # Threshold at 0.5 for multi-label

        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

# Calculate F1 score for each label
f1_per_label = f1_score(all_labels, all_preds, average=None)
print("F1 Score per label:")
for idx, score in enumerate(f1_per_label):
    print(f"Label {idx} F1 Score: {score:.4f}")



In [ ]:
model.load_state_dict(torch.load("roberta_emotion_model_Relu.pt", map_location=device))
model.eval()

# Predict emotions
def predict_emotions(text, model, tokenizer):
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=max_length, return_tensors="pt")
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        output = model(input_ids, attention_mask)
        probs = torch.sigmoid(output).cpu().numpy()[0]  # Convert logits to probabilities

    emotions = ["anger", "fear", "joy", "sadness", "surprise"]
    result = {emotion: round(float(prob), 2) for emotion, prob in zip(emotions, probs)}

    return result

# Example inference
text1 = "I am excited about this trip"
text2 = "William cat school bird"
text3 = "I am going there :)"
text4 = "I am doing it :("

print(f"Prediction of Meaningful Sentence: I am excited about this trip \n{predict_emotions(text1, model, tokenizer)} \nPrediction of Random Sentence: William cat school bird \n{predict_emotions(text2, model, tokenizer)}\nPrediction of neutral Sentence with emoji: I am going there :) \n{predict_emotions(text3, model, tokenizer)}\nPrediction of neutral Sentence with emoji: I am doing it :( \n{predict_emotions(text4, model, tokenizer)}")



In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
model.to(DEVICE)

Labels are Predicted for Test dataset

In [ ]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import pandas as pd
import numpy as np


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


MODEL_PATH = "roberta_emotion_model_Relu.pt"
NUM_LABELS = 5

model.load_state_dict(torch.load("roberta_emotion_model_Relu.pt", map_location=DEVICE))
model.to(DEVICE)
model.eval()


tokenizer = RobertaTokenizer.from_pretrained("roberta-base")


EMOTION_LABELS = ["anger", "fear", "joy", "sadness", "surprise"]

# Define max sequence length
MAX_LENGTH = 512

# Load test dataset
test_file = "//content/drive/My Drive/SemEval/track_a/test/eng.csv"
df = pd.read_csv(test_file)


TEXT_COLUMN = "text"
if TEXT_COLUMN not in df.columns:
    raise ValueError(f"Column '{TEXT_COLUMN}' not found in CSV!")

# Function to predict emotions with probability scores
def predict_emotions(text, model, tokenizer):
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt")

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        output = model(input_ids, attention_mask)
        probs = torch.sigmoid(output).cpu().numpy()[0]


    return {emotion: round(float(prob), 2) for emotion, prob in zip(EMOTION_LABELS, probs)}

# Performance on the test dataset
predictions = df[TEXT_COLUMN].fillna("").apply(lambda text: predict_emotions(text, model, tokenizer))

# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions.tolist())


In [ ]:
# Overwrite the existing label columns
df = df.drop(columns=[col for col in df.columns if col in EMOTION_LABELS], errors="ignore")
df = pd.concat([df[[TEXT_COLUMN]], predictions_df], axis=1)

# Save results
output_file = "test_predictions_with_probs.csv"
df.to_csv(output_file, index=False, sep=",", encoding="utf-8", na_rep="")

print(f"Predictions saved to {output_file}")

In [ ]:
testp_file = "/content/drive/My Drive/SemEval/test_predictions_with_probs.csv"
testp_texts, testp_labels = load_data(testp_file)
testp_encodings = tokenize_texts(testp_texts, tokenizer, max_length)
testp_dataset = EmotionDataset(testp_encodings, testp_labels, device="cuda")
testp_loader = DataLoader(testp_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
import torch
import shap
from transformers import RobertaTokenizer
import matplotlib.pyplot as plt

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

# Load trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RobertaClass(num_labels=5)
model.load_state_dict(torch.load("roberta_emotion_model_Relu.pt"), strict=False)
model.to(device)
model.eval()

class ModelWrapper:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def __call__(self, masked_texts):
        """Accept masked texts from SHAP, tokenize them, and return model outputs."""

        if isinstance(masked_texts, str):
            masked_texts = [masked_texts]
        # Tokenize text manually
        encoded_inputs = [self.tokenizer(text, padding="max_length", truncation=True, return_tensors="pt") for text in masked_texts]
        input_ids = torch.cat([x["input_ids"] for x in encoded_inputs]).to(device)
        attention_mask = torch.cat([x["attention_mask"] for x in encoded_inputs]).to(device)
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask)
            probs = torch.sigmoid(outputs)
        return probs.cpu().numpy()

# SHAP Masker for Token-based Models
masker = shap.maskers.Text(tokenizer, collapse_mask_token="[MASK]")

# Initialize SHAP Explainer
wrapper = ModelWrapper(model, tokenizer)
explainer = shap.Explainer(wrapper, masker, output_names=EMOTION_LABELS)

# Extract and clean sample texts from `test_loader`
sample_size = 3  # Adjust based on number of examples
sample_texts = []

for batch in testp_loader:
    input_ids = batch["input_ids"][:sample_size].to(device)

    # Decode the input IDs to text
    decoded_texts = [
        tokenizer.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        for ids in input_ids
    ]

    sample_texts.extend(decoded_texts)

    if len(sample_texts) >= sample_size:
        break

# Compute SHAP values with clean inputs
shap_values = explainer(sample_texts)
print(sample_texts)


for i in range(len(shap_values)):
    print(f"\n🔹 Sample {i+1}: {sample_texts[i]}")
    shap_output = shap_values[i]

    # Preserve spaces by replacing "Ġ" with an actual space
    shap_output.data = [token.replace("Ġ", " ") for token in shap_output.data]


    print(f"Processed Tokens: {shap_output.data}")

    shap.plots.text(shap_output)
